results of each tracking are in video files

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
import tensorflow as tf
import time
import os
import json
import kagglehub
from djitellopy import Tello
from datetime import datetime
from ultralytics import YOLO
from collections import defaultdict

path = kagglehub.model_download("tensorflow/ssd-mobilenet-v2/tensorFlow2/fpnlite-320x320")

In [17]:
# Models
modelCamera = tf.saved_model.load("saved_model")
modelDrone = YOLO("YOLO/yolov8s.pt")

# Function to perform human detection and distance estimation
def detect_and_estimate_distance(frame, threshold_area, model=modelCamera):
    input_tensor = tf.convert_to_tensor(frame)
    input_tensor = input_tensor[tf.newaxis,...]
    
    detections = model(input_tensor)

    boxes = detections['detection_boxes'][0].numpy()
    classes = detections['detection_classes'][0].numpy().astype(np.int32)
    scores = detections['detection_scores'][0].numpy()

    frame_height, frame_width, _ = frame.shape

    for i in range(len(scores)):
        if scores[i] > 0.5 and classes[i] == 1:  # Class 1 corresponds to 'person'
            ymin, xmin, ymax, xmax = boxes[i]
            (left, right, top, bottom) = (xmin * frame_width, xmax * frame_width, ymin * frame_height, ymax * frame_height)
            area = (right - left) * (bottom - top)
            
            # Draw bounding box
            cv2.rectangle(frame, (int(left), int(top)), (int(right), int(bottom)), (0, 255, 0), 2)
            
            # Check if area exceeds threshold
            if area > threshold_area:
                return True, frame
    
    return False, frame


def detect_objects_in_video(video_path, output_video_path, model=modelDrone):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    object_counts = defaultdict(int)

    # Get frame width and height
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))

    # Define the codec and create VideoWriter object for the output video
    out = cv2.VideoWriter(f'{output_video_path}/drone_with_detection.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30.0, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect objects in the frame
        results = model(frame)

        cv2.imshow("Frame", frame)

        for result in results:
            for box in result.boxes:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                object_counts[label] += 1

                # Draw bounding boxes and labels on the frame
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = box.conf[0]
                color = (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{label} {confidence:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Write the processed frame to the output video
        out.write(frame)
        frame_count += 1

    cap.release()
    out.release()
    return object_counts, frame_count

def calculate_percentage(object_counts, frame_count):
    percentages = {obj: (count / frame_count) * 100 for obj, count in object_counts.items()}
    return percentages

def save_json(data, output_folder):
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
    
   # Sort data by percentages in descending order
    sorted_data = dict(sorted(data.items(), key=lambda item: item[1], reverse=True))

    # Define the output JSON file path
    json_file_path = os.path.join(output_folder, "object_detection_results.json")

    # Save sorted data as JSON
    with open(json_file_path, "w") as json_file:
        json.dump(sorted_data, json_file, indent=4)


def run_video_object_detection(video_path, output_folder, model=modelDrone):

    # Perform object detection and count objects
    object_counts, frame_count = detect_objects_in_video(video_path, output_folder, model)
    
    # Calculate percentages
    object_percentages = calculate_percentage(object_counts, frame_count)

    # Prepare data for JSON
    data = {obj: f"{percentage:.2f}%" for obj, percentage in object_percentages.items()}

    # Save the results to a JSON file
    save_json(data, output_folder)
    
    print(f"Object detection results and video saved to {output_folder}/")


def list_connected_devices(max_devices=10):
    available_devices = []
    for device_index in range(max_devices):
        cap = cv2.VideoCapture(device_index)
        if cap.isOpened():
            available_devices.append(device_index)
            cap.release()
    return available_devices
#print("Connected video devices:", list_connected_devices())

def tello_fly_record():

    # Create a Tello object
    tello = Tello()

    # Connect to the Tello drone
    tello.connect()

    # Print the battery level
    battery = tello.get_battery()
    print(f"Battery level: {battery}%")

    # Get current date for folder name
    current_date = datetime.now().strftime("%Y-%m-%d")
    if not os.path.exists(f'{current_date}'):
        os.makedirs(f'{current_date}')

    # Start video stream
    tello.streamon()

    # Initialize the video writer variable
    frame_read = tello.get_frame_read()
    time.sleep(2)
    frame = frame_read.frame
    height, width, _ = frame.shape

    video = None  # Initialize video writer variable
    
    print("Starting video capture. Press 'q' to quit.")

    # Take off
    tello.takeoff()

    # Initialize the video writer once the drone takes off
    video = cv2.VideoWriter(f'{current_date}/drone.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30.0, (width, height))

    try:
        # Move up by 1.5 meter
        tello.move_up(150)
        time.sleep(1)

        # Move forward by 1 meter
        tello.move_forward(100)
        time.sleep(1)

        # Move back by 1 meter
        tello.move_back(100)
        time.sleep(1)

    except Exception as e:
        print(f"An error occurred during drone movement: {e}")
        

    while tello.get_height() > 0:  # Continue recording while the drone is airborne
        # Get the current frame from the drone
        frame = frame_read.frame
        
        # Write the frame to the video file
        if video:
            video.write(frame)
        
        # Display the frame (optional)
        cv2.imshow("Tello Video Stream", frame)
        
        # Check for 'q' key press to quit early
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Stop recording when the drone lands
    if video:
        video.release()  
        cv2.destroyAllWindows() 

    # Land the drone
    tello.land()

    # Stop video stream
    tello.streamoff()

    # Disconnect from the drone (optional, depending on your Tello library version)
    #tello.end()

    # Perform object detection on the recorded video
    run_video_object_detection(f'{current_date}/drone.avi', current_date, model=modelDrone)

def main(camera):
    # Access webcam and process frames
    cap = cv2.VideoCapture(camera)
    threshold_area = 50000  # Define an appropriate threshold area

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        is_close, processed_frame = detect_and_estimate_distance(frame, threshold_area, model=modelCamera)

        cv2.imshow("Frame", processed_frame)

        if is_close:
            # Run drone.py
            # Save frame to folder with name of current date
            current_date = datetime.now().strftime("%Y-%m-%d")
            if not os.path.exists(f'{current_date}'):
                os.makedirs(f'{current_date}')
            frame_path = os.path.join(f'{current_date}', f"frame_{datetime.now().strftime('%H-%M-%S')}.jpg")
            cv2.imwrite(frame_path, processed_frame)

            # Initialize the drone
            tello_fly_record()

            break
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run the main function
main(0)

#if __name__ == "__main__":
    #main()

[INFO] tello.py - 129 - Tello instance was initialized. Host: '192.168.10.1'. Port: '8889'.
INFO:djitellopy:Tello instance was initialized. Host: '192.168.10.1'. Port: '8889'.
[INFO] tello.py - 438 - Send command: 'command'
INFO:djitellopy:Send command: 'command'
[INFO] tello.py - 462 - Response command: 'ok'
INFO:djitellopy:Response command: 'ok'
[INFO] tello.py - 438 - Send command: 'streamon'
INFO:djitellopy:Send command: 'streamon'
[INFO] tello.py - 462 - Response streamon: 'ok'
INFO:djitellopy:Response streamon: 'ok'


Battery level: 38%


ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:decode_slice_header error
ERROR:libav.h264:no frame!
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:decode_slice_header error
ERROR:libav.h264:no frame!
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:decode_slice_header error
ERROR:libav.h264:no frame!
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:decode_slice_header error
ERROR:libav.h264:no frame!
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:decode_slice_header error
ERROR:libav.h264:no frame!
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:non-existing PPS 0 referenced
ERROR:libav.h264:decode_slice_header error
ERROR:libav.h264:no frame!
ERROR:libav.h264

Starting video capture. Press 'q' to quit.


[INFO] tello.py - 462 - Response takeoff: 'ok'
INFO:djitellopy:Response takeoff: 'ok'
[INFO] tello.py - 438 - Send command: 'up 150'
INFO:djitellopy:Send command: 'up 150'
[INFO] tello.py - 462 - Response up 150: 'ok'
INFO:djitellopy:Response up 150: 'ok'
[INFO] tello.py - 438 - Send command: 'forward 100'
INFO:djitellopy:Send command: 'forward 100'
[INFO] tello.py - 462 - Response forward 100: 'ok'
INFO:djitellopy:Response forward 100: 'ok'
[INFO] tello.py - 438 - Send command: 'back 100'
INFO:djitellopy:Send command: 'back 100'
[INFO] tello.py - 462 - Response back 100: 'ok'
INFO:djitellopy:Response back 100: 'ok'
[INFO] tello.py - 438 - Send command: 'land'
INFO:djitellopy:Send command: 'land'
[WARNING] tello.py - 448 - Aborting command 'land'. Did not receive a response after 7 seconds
[INFO] tello.py - 438 - Send command: 'land'
INFO:djitellopy:Send command: 'land'
[INFO] tello.py - 462 - Response land: 'ok'
INFO:djitellopy:Response land: 'ok'
[INFO] tello.py - 438 - Send command:


0: 480x640 2 couchs, 1 laptop, 332.2ms
Speed: 2.8ms preprocess, 332.2ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 couchs, 1 laptop, 295.4ms
Speed: 2.7ms preprocess, 295.4ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 couchs, 1 laptop, 293.4ms
Speed: 2.4ms preprocess, 293.4ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 couchs, 1 laptop, 297.7ms
Speed: 2.1ms preprocess, 297.7ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 couchs, 1 laptop, 299.6ms
Speed: 1.9ms preprocess, 299.6ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 couchs, 1 laptop, 302.3ms
Speed: 2.4ms preprocess, 302.3ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 couchs, 1 laptop, 296.0ms
Speed: 1.8ms preprocess, 296.0ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 couchs, 1 laptop, 29

In [ ]:
from wordpress import API
import os

# Configuration
wordpress_url = "https://your-wordpress-site.com"
consumer_key = "your-consumer-key"
consumer_secret = "your-consumer-secret"

# File paths
video_path = "path/to/your/video.mp4"
image_path = "path/to/your/image.jpg"

# Initialize the WordPress API client
wp = API(
    url=wordpress_url,
    consumer_key=consumer_key,
    consumer_secret=consumer_secret,
    api="wp-json",
    version="v2"
)

# Function to upload media
def upload_media(file_path, media_type):
    file_name = os.path.basename(file_path)
    with open(file_path, 'rb') as file:
        files = {
            'file': (file_name, file, media_type),
        }
        response = wp.post('media', files=files)
        media_id = response['id']
        media_link = response['guid']['rendered']
        return media_id, media_link

# Upload video
video_id, video_link = upload_media(video_path, "video/mp4")

# Upload image
image_id, _ = upload_media(image_path, "image/jpeg")

# Create a new post
post_data = {
    "title": "New Post with Video",
    "content": "This post contains a video and an image.",
    "status": "publish",  # Change to 'draft' if you don't want to publish immediately
    "featured_media": image_id,  # Set the uploaded image as featured image
    "meta": {
        "video": video_link  # Save the video link in the custom field 'video'
    }
}

post = wp.post("posts", data=post_data)

# Print the link to the new post
print("New post created:", post['link'])
